In [ ]:
!pip install -U bitsandbytes transformers datasets peft accelerate evaluate trl unsloth python-docx

In [ ]:
import os
import json
import re
import torch

from datasets import Dataset
from sentence_transformers import SentenceTransformer, util

In [ ]:
# 1) Load raw cards & fully‑marked outputs
with open('input_10.json','r',encoding='utf-8') as f_in, \
     open('output_10.json','r',encoding='utf-8') as f_out:
    inputs  = json.load(f_in)
    outputs = json.load(f_out)
assert len(inputs)==len(outputs), "Mismatched lengths"

# 2) Extract spans from existing <ul>,<h>,<e> markup
def extract_spans(marked_text):
    spans = []
    for tag in ("ul","h","e"):
        for m in re.finditer(rf"<{tag}>(.*?)</{tag}>", marked_text, flags=re.DOTALL):
            spans.append({
                "start": m.start(1),
                "end":   m.end(1),
                "type":  tag
            })
    spans.sort(key=lambda s: (s["start"], -s["end"]))
    return spans

# 3) Prompt prefix, with full debate‑card context
span_instructions = (
    "You are a meticulous AI assistant specialized in identifying markup spans in competitive debate cards.\n\n"
    "**Debate Cards Background:**\n"
    "- In debate, speakers use \"cards\": highly structured excerpts from sources.\n"
    "- Each card has three parts: a brief *tag* (the point), a *cite* (source reference), and a body of text.\n"
    "- To prepare, debaters mark the body with underlines for evidence, highlights for exact phrasing, and emphasis for tonal cues.\n\n"
    "**Understanding the Tags:**\n"
    "- Underlining `<ul>…</ul>`: marks relevant evidence or claims.\n"
    "- Highlighting `<h>…</h>`: exact words the speaker will read aloud (always inside an underline).\n"
    "- Emphasis `<e>…</e>`: tonal shifts or emotional weight (inside highlights).\n\n"
    "**Strict Span Rules:**\n"
    "1. Each span must align to whole‐word boundaries—never split a word.\n"
    "2. Output ONLY a valid JSON array of span objects, each with keys `start`, `end`, `type`.\n"
    "3. Ensure the array begins with `[` and ends with `]`—no extra text.\n\n"
)

# 4) Build training examples
examples = []
for inp, out in zip(inputs, outputs):
    spans = extract_spans(out["text_body"])
    prompt = (
        span_instructions
      + f"Tag: {inp['tag']}\n"
      + f"Cite: {inp['cite']}\n"
      + f"Text:\n{inp['text_body']}\n\n"
        "Output JSON spans:\n"
    )
    examples.append({"text": prompt + json.dumps(spans, ensure_ascii=False)})

dataset = Dataset.from_list(examples)
print(f"Prepared span‑prediction dataset with {len(dataset)} examples")

In [ ]:
max_seq_length = 4096
dtype           = None   # auto fp16/bf16
load_in_4bit    = True

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
markup_tokens = ['<ul>','</ul>','<h>','</h>','<e>','</e>']
tokenizer.add_special_tokens({
    'additional_special_tokens': markup_tokens,
    'pad_token': '[PAD]'
})
model.resize_token_embeddings(len(tokenizer))
tokenizer.padding_side = 'right'

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="debate-finetune",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,
    args               = training_args,
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("debate-finetune", safe_serialization=True)
tokenizer.save_pretrained("debate-finetune")
print("Adapters and tokenizer saved to ./debate-finetune")

In [ ]:
# Cell 10 – Inference helper with robust fallback for spans JSON

import re
import json

def get_markup_spans(card_data):
    prompt = (
        span_instructions
      + f"Tag: {card_data['tag']}\n"
      + f"Cite: {card_data['cite']}\n"
      + f"Text:\n{card_data['text_body']}\n\n"
        "Output JSON spans:\n"
    )

    # Tokenize & generate
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_seq_length
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        input_ids      = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 2048,            # ↑ increased to 2048
        pad_token_id   = tokenizer.pad_token_id,
        eos_token_id   = tokenizer.eos_token_id,
        do_sample      = False,
    )

    # Decode only the generated suffix
    span_tokens = output_ids[0, prompt_len:]
    candidate   = tokenizer.decode(
        span_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()

    # 1) Try to extract a balanced JSON array via bracket‑matching
    first = candidate.find('[')
    if first == -1:
        raise ValueError(f"No '[' found in model output:\n{candidate}")

    count = 0
    last  = None
    for idx, ch in enumerate(candidate[first:], start=first):
        if ch == '[':
            count += 1
        elif ch == ']':
            count -= 1
            if count == 0:
                last = idx
                break

    # If no closing bracket found, append one
    if last is None:
        candidate += ']'
        last = len(candidate)-1

    json_str = candidate[first:last+1]

    # 2) Attempt normal JSON parsing
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # 3) Fallback: grab all standalone `{…}` objects via regex
        objs = re.findall(r'\{[^}]+\}', json_str)
        spans = []
        for o in objs:
            try:
                spans.append(json.loads(o))
            except json.JSONDecodeError:
                continue
        if not spans:
            raise ValueError(f"Could not parse any spans from:\n{json_str}")
        return spans

In [ ]:
def adjust_spans_to_word_boundaries(text, spans):
    adjusted = []
    L = len(text)
    for s in spans:
        st, ed, typ = max(0, min(s["start"], L)), max(0, min(s["end"], L)), s["type"]
        # expand to words
        while st>0 and re.match(r'\w', text[st-1]): st -= 1
        while ed<L and re.match(r'\w', text[ed]):   ed += 1
        if ed>st:
            adjusted.append({"start": st, "end": ed, "type": typ})
    adjusted.sort(key=lambda s: (s["start"], -s["end"]))
    return adjusted

def enforce_spans_within_ul(spans):
    ul_spans = [s for s in spans if s["type"]=="ul"]
    kept     = []
    for s in spans:
        if s["type"]=="ul":
            kept.append(s)
        else:
            # find best overlap with a ul
            best_len = 0
            best_seg = None
            for ul in ul_spans:
                os = max(s["start"], ul["start"])
                oe = min(s["end"],   ul["end"])
                if oe>os and (oe-os)>best_len:
                    best_len = oe-os
                    best_seg = (os, oe)
            if best_seg:
                kept.append({"start": best_seg[0], "end": best_seg[1], "type": s["type"]})
    kept.sort(key=lambda s: (s["start"], -s["end"]))
    return kept

def insert_markup(text, spans):
    # 1) snap to words
    spans = adjust_spans_to_word_boundaries(text, spans)
    # 2) enforce h/e only inside ul
    spans = enforce_spans_within_ul(spans)

    # 3) build open/close events
    events = []
    for s in spans:
        events.append((s["start"], 0, f"<{s['type']}>"))
        events.append((s["end"],   1, f"</{s['type']}>"))

    # 4) sort events: pos↑, open before close, ul→h→e / close e→h→ul
    def sort_key(ev):
        pos, kind, tag = ev
        t = tag.strip("</>")
        order = {"ul":0,"h":1,"e":2}[t]
        return (pos, kind, order if kind==0 else 2-order)
    events.sort(key=sort_key)

    # 5) splice
    out, last = [], 0
    for pos, _, tag in events:
        out.append(text[last:pos]); out.append(tag)
        last = pos
    out.append(text[last:])
    return "".join(out)

def format_debate_card(card_data):
    spans     = get_markup_spans(card_data)
    formatted = insert_markup(card_data["text_body"], spans)
    return {
        "tag":       card_data["tag"],
        "cite":      card_data["cite"],
        "text_body": formatted
    }

In [ ]:
import json, re
from docx import Document
from docx.enum.text import WD_COLOR_INDEX

def add_formatted_paragraph(doc, text_with_markup):
    p     = doc.add_paragraph()
    parts = re.split(r'(<ul>|</ul>|<h>|</h>|<e>|</e>)', text_with_markup)
    in_ul = in_h = in_e = False
    for part in parts:
        if part == '<ul>':      in_ul = True
        elif part == '</ul>':   in_ul = False
        elif part == '<h>':     in_h  = True
        elif part == '</h>':    in_h  = False
        elif part == '<e>':     in_e  = True
        elif part == '</e>':    in_e  = False
        elif part:
            run = p.add_run(part)
            if in_ul: run.underline = True
            if in_h:  run.font.highlight_color = WD_COLOR_INDEX.YELLOW
            if in_e:  run.bold = True

def convert_card_json_to_docx(card_input, docx_output):
    """
    card_input: either a path to a JSON file or a dict containing the card
    docx_output: path where the .docx will be saved
    """
    # Load the card
    if isinstance(card_input, str):
        card = json.load(open(card_input, encoding='utf-8'))
    else:
        card = card_input

    # Build the document
    doc = Document()
    doc.add_heading(card['tag'], level=4)
    doc.add_paragraph(card['cite'])
    add_formatted_paragraph(doc, card['text_body'])
    doc.save(docx_output)
    print(f"Wrote formatted card to {docx_output}")

# ────────────────
# Example usage (run this in another cell):
# formatted = format_debate_card(new_card)
# convert_card_json_to_docx(formatted, "output.docx")

In [ ]:
# Cell 14 – Interactive prompt & export to .docx with single‑line text_body

import re

# 1) Prompt for tag and cite
tag = input("Enter the debate card tag (e.g. “Climate Change - Major Impact”):\n> ").strip()
cite = input("\nEnter the citation string (e.g. “Smith, 2023 (Environmental Journal, pp. 45-67)”):\n> ").strip()

# 2) Prompt for the text_body as a single line
text_body = input("\nEnter the text_body (entire content on one line):\n> ").strip()

# 3) Prompt for desired Word filename (no extension) and clean it
raw_name = input("\nEnter desired filename for the Word doc (no extension):\n> ").strip()
clean_name = re.sub(r'[^A-Za-z0-9_-]', '', raw_name)
if not clean_name:
    clean_name = re.sub(r'[^A-Za-z0-9_-]', '_', tag) or "debate_card"
docx_path = f"{clean_name}.docx"

# 4) Build and format the card
card = {"tag": tag, "cite": cite, "text_body": text_body}
formatted = format_debate_card(card)

# 5) Save to Word
convert_card_json_to_docx(formatted, docx_path)

# 6) Completion message
print(f"\n✅ Your formatted debate card has been saved to: {docx_path}")